# PySeis Wiggle Plots Demo
This notebook demonstrates how to plot wiggle traces using PySeis with Matplotlib, Bokeh, and Plotly.

In [ ]:
%pip install matplotlib bokeh plotly segyio

In [ ]:
# Install pyseiskit from PyPI
%pip install pyseiskit
# Install for local development
# %pip install -e ..

## 1A. Option 1: Create Simple Synthetic Data
Run this cell to generate a small 2D array representing 5 seismic traces, each with 100 time samples.

In [ ]:
import numpy as np

# 100 samples, 5 traces
data = np.random.randn(100, 5) 
times = np.arange(100)
offsets = np.arange(5, dtype=float)


## 1B. Option 2: Load Real `.su` or `.sgy` Data
Run this cell instead if you have the file `tac-204RL239.su` in the same folder as this notebook.

In [ ]:
import segyio
from seismic_reader import read_seismic_file

filename = 'tac-204RL239.su'

# We only want to plot a single gather to avoid rendering thousands of traces.
# FieldRecord corresponds to the 'fldr' header in SU/SEGY.
gather_key = segyio.TraceField.FieldRecord
gather_index = 50  # Change this to a valid fldr number from your dataset!

try:
    data, offsets, times = read_seismic_file(filename, gather_key=gather_key, gather_index=gather_index)
    print(f"Loaded {data.shape[1]} traces for fldr {gather_index} with {data.shape[0]} samples each.")
except FileNotFoundError:
    print(f"File '{filename}' not found. Please place it in the 'demos' folder or stick to Option 1A.")
except ValueError as e:
    print(f"Error: {e}")

## 2. Process Data for Wiggles
Regardless of how you loaded the data above, we now pass it through PySeis.

In [ ]:
from pyseiskit import sourceData

scaled_data = sourceData.rescaleDataForWiggle(data, offsets, overlap=1.0)
line_data = sourceData.wiggleLinesDataFactory(scaled_data, offsets, times)
patch_data = sourceData.wigglePatchesDataFactory(scaled_data, offsets, times, fill_mode='positive')

## 3. Plotting with Matplotlib

In [ ]:
import matplotlib.pyplot as plt

matplotlib_figure, matplotlib_axes = plt.subplots(figsize=(6, 4))

# Plot filled areas
for x, y in zip(patch_data['xs'], patch_data['ys']):
    matplotlib_axes.fill(x, y, color='black')

# Plot trace lines
for x, y in zip(line_data['xs'], line_data['ys']):
    matplotlib_axes.plot(x, y, color='black', linewidth=0.5)

matplotlib_axes.invert_yaxis() # Time goes down
plt.show()

## 4. Plotting with Bokeh

In [ ]:
from bokeh.plotting import figure, show, output_notebook
output_notebook()

bokeh_plot = figure(width=600, height=400, y_range=(times[-1], times[0]))
bokeh_plot.multi_line(**line_data, color='black', line_width=0.5)
bokeh_plot.patches(**patch_data, color='black', line_width=0)

show(bokeh_plot)

## 5. Plotting with Plotly

In [ ]:
import plotly.graph_objects as go

plotly_figure = go.Figure()

for x, y in zip(line_data['xs'], line_data['ys']):
    plotly_figure.add_trace(go.Scatter(x=x, y=y, mode='lines', line=dict(color='black', width=0.5), showlegend=False))

for x, y in zip(patch_data['xs'], patch_data['ys']):
    plotly_figure.add_trace(go.Scatter(x=x, y=y, fill='toself', fillcolor='black', line=dict(width=0), showlegend=False))

plotly_figure.update_layout(yaxis=dict(autorange='reversed'), height=500)
plotly_figure.show()